# Pediatric chest X-ray classification: NORMAL vs PNEUMONIA
**Applied Machine Learning – Advanced · Edoardo Buttazzi · second submission**

This notebook only *orchestrates* the experiment; all logic lives in the `src/` package
and every hyperparameter or path lives in `configs/default.yaml`.

| Section | Question answered |
|---|---|
| 1. Setup | environment, seeds, data location |
| 2. Dataset & EDA | what is in the data, how the splits are built, which shortcuts exist |
| 3. Baselines | how far do trivial / non-deep models go? |
| 4. Screening experiments | which design choices matter (architecture, imbalance, padding, LR)? |
| 5. Model selection | choice made **on validation only** |
| 6. Final training | selected configuration, 512 px, 3 seeds |
| 7. Test evaluation | ROC / PR, confusion matrix at a threshold frozen on validation |
| 8. Error analysis & explainability | confident errors, Grad-CAM, test-wide padding ablation |
| 9. Export | LaTeX tables / macros / figures used by the report |

**Changes with respect to the first submission**
* patient-grouped validation split + removal of exact duplicates (leakage control);
* classical and deep baselines, one-factor-at-a-time ablations, multiple seeds;
* the imbalance strategy is explicit: `pos_weight` is either used (`weighted_bce`) or absent,
  the focal loss has a real class weight `alpha_t` (the old scalar 0.85 weighted both classes equally);
* the decision threshold is chosen on validation and frozen before the test set is used;
* the same checkpoint (identified by SHA-256) produces metrics, curves and Grad-CAM maps.

## 1. Setup
**Colab:** the code is copied from Drive to the fast local disk (`/content/aml_pneumonia`), the dataset is
extracted there, and `runs/` and `results/` are links to Drive, so checkpoints and results survive a disconnection.
**Locally:** `pip install -r requirements.txt` and skip the next cell.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_PROJECT = "/content/drive/MyDrive/aml_pneumonia"   # code + saved outputs
    ZIP = "/content/drive/MyDrive/ChestXRay2017.zip"          # dataset archive
    LOCAL = "/content/aml_pneumonia"
    import shutil
    shutil.copytree(DRIVE_PROJECT, LOCAL, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("runs", "results", "data", "__pycache__"))
    os.makedirs(f"{DRIVE_PROJECT}/runs", exist_ok=True); os.makedirs(f"{DRIVE_PROJECT}/results", exist_ok=True)
    for d in ("runs", "results"):
        if not os.path.exists(f"{LOCAL}/{d}"):
            os.symlink(f"{DRIVE_PROJECT}/{d}", f"{LOCAL}/{d}")
    os.chdir(LOCAL)
    !pip install -q -r requirements.txt
    if not os.path.isdir("data/chest_xray"):
        !unzip -q "{ZIP}" -d data -x "__MACOSX/*"
    !nvidia-smi --query-gpu=name,memory.total --format=csv
print("working dir:", os.getcwd())

In [ ]:
from pathlib import Path
import json, yaml
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage

from src.config import load_yaml, build_experiment_config
from src.data import build_manifest, make_splits, split_summary
from src.utils import set_seed, get_device
from src import plots

CFG = load_yaml("configs/default.yaml")
PLAN = load_yaml("configs/experiments.yaml")
RESULTS = Path(CFG["paths"]["results_dir"]); FIG = RESULTS / "figures"; FIG.mkdir(parents=True, exist_ok=True)
set_seed(CFG["seed"])
DEVICE = get_device()
print("device:", DEVICE, "| torch", torch.__version__)
print(yaml.safe_dump(CFG, sort_keys=False))

## 2. Dataset and exploratory analysis
Kermany et al. (2018), pediatric anteroposterior radiographs, Guangzhou Women and Children's
Medical Center. The archive ships `train/` and `test/`; no validation folder.

The manifest records label, bacterial/viral subtype, a conservative **patient key** derived from
filenames, native size, colour mode and SHA-256. It is cached in `results/manifest.csv`.
The full audit (intensity statistics, near-duplicate screen, representative images) is in the report, Section 3.

In [ ]:
manifest = build_manifest(CFG["paths"]["data_dir"], RESULTS / "manifest.csv")
print(f"{len(manifest):,} images")
display(pd.crosstab(manifest.source_split, manifest.subtype, margins=True))

### 2.1 Exact duplicates and patient keys

In [ ]:
dup = manifest[manifest.duplicated("sha256", keep=False)]
print(f"duplicate groups: {dup.sha256.nunique()}, records: {len(dup)}, redundant copies: {len(dup) - dup.sha256.nunique()}")
print("groups spanning train/test:", (dup.groupby("sha256").source_split.nunique() > 1).sum())
per_patient = manifest[manifest.source_split == "train"].groupby("patient_id").size()
print(f"development patient keys: {len(per_patient):,}; keys with >1 image: {(per_patient > 1).sum():,}; "
      f"max images per key: {per_patient.max()}")

### 2.2 Split composition
Validation is built with `StratifiedGroupKFold` on the patient key (one fold = 20 %), so no
patient key appears in both training and validation. The image-level split of the first
submission is kept as experiment `ref_image_split` to *measure* the effect of leakage.

In [ ]:
splits = make_splits(manifest, CFG)
display(split_summary(splits).round(1))

### 2.3 Geometric shortcut
Native size differs strongly between classes. If size alone separates the labels, a high AUC is not by itself evidence of radiological understanding.

In [ ]:
from sklearn.metrics import roc_auc_score
for name, df in splits.items():
    print(f"{name:5s}  AUC of score -width: {roc_auc_score(df.target, -df.width):.3f}")
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
for cls, c, lab in ((0, plots.SERIES[0], "NORMAL"), (1, plots.SERIES[1], "PNEUMONIA")):
    d = manifest[manifest.target == cls]
    ax[0].hist(d.width, bins=50, alpha=0.6, color=c, label=lab)
    ax[1].hist(d.width / d.height, bins=50, alpha=0.6, color=c, label=lab)
ax[0].set(xlabel="native width (px)", ylabel="images"); ax[1].set(xlabel="aspect ratio W/H"); ax[0].legend()
plt.tight_layout(); plt.show()

## 3. Baselines
All baselines use the same split, the same validation-selected threshold rule and the same metrics.
* **majority** – constant score; sets the floor (AUC 0.5, accuracy = prevalence).
* **geometry** – logistic regression on native width, height and aspect ratio *only*.
* **pixels_lr** – logistic regression on 64×64 thumbnails (no deep features).

The deep baselines (from-scratch CNN, DenseNet linear probe, ResNet50) are part of the screening grid.

In [ ]:
from src.baselines import run_baselines
base = run_baselines(splits, {**CFG, "experiment": {}}, Path(CFG["paths"]["output_dir"]) / "baselines")
pd.DataFrame({k: {"val AUC": v["val"]["auc"], "test AUC": v["test"]["auc"],
                  "val bal.acc": v["val"]["at_selected"]["balanced_accuracy"]} for k, v in base.items()}).T.round(3)

## 4. Screening experiments
One factor at a time with respect to the reference configuration (`configs/experiments.yaml`):

| Group | Question |
|---|---|
| Q0 | how much did image-level leakage inflate validation? |
| Q1 | from scratch vs linear probe vs fine-tuning; DenseNet121 vs ResNet50 |
| Q2 | imbalance: plain BCE, `pos_weight`, sampler, class-weighted focal loss |
| Q3 | padding: reflection (original) vs constant vs stretch (motivated by Grad-CAM) |
| Q4 | learning-rate sensitivity |

Screening runs at 384 px, 8 epochs, seed 42 (≈ 10 min/run on a T4). Finished runs are skipped,
so the cell can be re-executed after a Colab disconnection.

In [ ]:
from run_experiment import run_one
for exp in PLAN["experiments"]:
    for seed in PLAN["seeds_screening"]:
        run_one(exp, seed, stage="screening")

## 5. Model selection (validation only)
Rule fixed in advance: highest mean validation ROC-AUC; ties broken by validation balanced accuracy.
The test set has not been used for any decision up to this point.

In [ ]:
!python summarize.py
scr = pd.read_csv(RESULTS / "summary_screening.csv")
display(scr.sort_values("val_auc_mean", ascending=False)
           [["group", "experiment", "val_auc_mean", "val_ap_mean", "val_bacc_mean", "minutes_mean"]].round(4))
display(IPImage(str(FIG / "screening_val_auc.png")))

In [ ]:
ranked = scr.sort_values(["val_auc_mean", "val_bacc_mean"], ascending=False)
SELECTED = ranked.experiment.iloc[0]
FINALISTS = list(dict.fromkeys([SELECTED, "ref_patient_split"]))  # always keep the reference for comparison
print("selected:", SELECTED, "| finalists:", FINALISTS)

## 6. Final training
Finalists are re-trained at 512 px with three seeds; mean ± sd is reported.

In [ ]:
for exp in FINALISTS:
    for seed in PLAN["seeds_final"]:
        run_one(exp, seed, stage="final")
!python summarize.py

## 7. Test evaluation
The operating point is the highest threshold reaching validation sensitivity ≥ 0.95
(`evaluation.threshold_rule`). It is frozen and applied once to the test set.

In [ ]:
for f in ("learning_curves_final", "roc_pr_test", "confusion_test"):
    display(IPImage(str(FIG / f"{f}.png")))
fin = pd.read_csv(RESULTS / "summary_final.csv")
display(fin[["experiment", "n_seeds", "threshold_mean", "test_auc_mean", "test_auc_std",
             "test_sens_mean", "test_spec_mean", "test_f1_mean"]].round(4))

## 8. Error analysis and explainability
### 8.1 Most confident errors

In [ ]:
from src.gradcam import select_cases, analyse_cases, gradcam_rows
from src.models import build_model

run_dir = Path(CFG["paths"]["output_dir"]) / "final" / SELECTED / f"seed{PLAN['seeds_final'][0]}"
run_cfg = load_yaml(run_dir / "config_resolved.yaml")
metrics = json.loads((run_dir / "metrics.json").read_text())
THR = metrics["val"]["at_selected"]["threshold"]
print("checkpoint sha256:", metrics["checkpoint_sha256"][:16], "| threshold:", round(THR, 4))

test_pred = pd.read_csv(run_dir / "test_predictions.csv")
cases = select_cases(test_pred, THR, k=2)
cases["image_path"] = [str(Path(CFG["paths"]["data_dir"]) / p) for p in cases.rel_path]
display(cases[["outcome", "rel_path", "subtype", "width", "height", "prob"]])
plots.error_gallery(cases, CFG["paths"]["data_dir"], FIG / "error_gallery")
display(IPImage(str(FIG / "error_gallery.png")))

### 8.2 Grad-CAM with padding neutralisation (same checkpoint)

In [ ]:
model = build_model({**run_cfg, "model": {**run_cfg["model"], "pretrained": False}}).to(DEVICE)
model.load_state_dict(torch.load(run_dir / "best.pt", map_location=DEVICE)); model.eval()
rows = gradcam_rows(model, cases[cases.outcome.isin(["TP", "TN", "FP", "FN"])].groupby("outcome").head(1),
                    run_cfg, DEVICE, THR)
plots.gradcam_grid(rows, FIG / "gradcam_cases")
display(IPImage(str(FIG / "gradcam_cases.png")))

### 8.3 Test-wide padding analysis
Instead of four illustrative cases, padding attribution mass and the effect of neutralising
padding are measured on **all** test images. A model that ignores padding would lie on the
diagonal of the right panel and below the diagonal of the left panel.

In [ ]:
test_df = splits["test"]
pad = analyse_cases(model, test_df, run_cfg, DEVICE, THR)
pad.to_csv(RESULTS / f"padding_analysis_{SELECTED}.csv", index=False)
plots.padding_summary_plot(pad, FIG / "padding_summary")
display(IPImage(str(FIG / "padding_summary.png")))
summary = {
    "median padding mass": pad.pad_mass_orig.median(),
    "median padding area": pad.pad_area.median(),
    "share with enrichment > 1": (pad.pad_enrichment_orig > 1).mean(),
    "predictions flipped by neutralisation": (pad.pred_orig != pad.pred_neutral).mean(),
    "median |delta p|": (pad.p_orig - pad.p_neutral).abs().median(),
}
pd.Series(summary).round(3)

## 9. Export for the report
`summarize.py` writes `results/tables/*.tex` (booktabs tables and `macros.tex`) and `results/figures/*.pdf`; the report `\input`s them, so no number is copied by hand.

In [ ]:
!python summarize.py
!ls results/tables results/figures